# AfriVoices ASR - Colab Pro low-RAM Whisper-small Round 6B run

Recommended Colab runtime: **A100 GPU + High-RAM**. H100 also works. Do not use TPU for this notebook.

This rewrite avoids the memory failure from the Kaggle notebook by caching each training clip as a small FLAC file plus a JSONL manifest, then computing Whisper log-mel features inside the batch collator. It never keeps all audio arrays or all feature tensors in system RAM.

Target experiment: continue from the best previous Whisper-small checkpoint with a stronger ANV Somali/Maxatire and unscripted-speech mix, while keeping the low-RAM cache design. A public score around 0.31-0.32 is not guaranteed, but this is the right shape of run for trying to move far below the current ~0.75 result.


In [ ]:
# CELL 1 - Runtime, Drive, installs
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess

DRIVE_DIR = '/content/drive/MyDrive/afrivoices_colab_round6b'
LOCAL_DIR = '/content/afrivoices_round6b'
for d in [DRIVE_DIR, LOCAL_DIR, f'{DRIVE_DIR}/audio_cache', f'{DRIVE_DIR}/manifests', f'{DRIVE_DIR}/checkpoints', f'{DRIVE_DIR}/outputs']:
    os.makedirs(d, exist_ok=True)

# Colab Python 3.12 can mix preinstalled NumPy 2.x wheels with packages that
# still import NumPy 1.x ABI extensions. Pin and reinstall the numeric stack
# first, then install the ASR dependencies against that stack.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
    'numpy==1.26.4', 'pandas==2.2.2', 'pyarrow==16.1.0'
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers==4.46.3', 'datasets==2.20.0', 'accelerate>=0.26.0',
    'evaluate', 'jiwer', 'soundfile', 'librosa', 'pydub',
    'huggingface_hub>=0.21', 'kagglehub', 'psutil'
], check=True)

print('Dependencies installed.')
print('IMPORTANT: if this is the first time running Cell 1 in this runtime, restart the runtime now, then continue with Cell 2.')


In [ ]:
# CELL 2 - Imports, secrets, and global config
from google.colab import userdata
import gc, glob, io, json, math, os, random, re, shutil, tarfile, tempfile, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import evaluate
import kagglehub
import librosa
import numpy as np
import pandas as pd
import psutil
import soundfile as sf
import torch
from huggingface_hub import HfApi, hf_hub_download, list_repo_files, login
from pydub import AudioSegment
from torch.utils.data import Dataset as TorchDataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperForConditionalGeneration, WhisperProcessor

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets before running this notebook.')
login(token=HF_TOKEN)

# Optional for kagglehub downloads. Current Kaggle auth only needs the API key.
KAGGLE_KEY = userdata.get('KAGGLE_KEY') or userdata.get('KAGGLE_API_TOKEN')
if KAGGLE_KEY:
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'openai/whisper-small'
# Start from your strongest previous repo if it exists. Fall back to base if loading fails.
START_FROM = 'Ash11/afrivoices-whisper-small-all6'
ROUND_REPO_NAME = 'afrivoices-whisper-small-colab-round6b'

SAMPLE_RATE = 16000
MAX_AUDIO_SAMPLES = 480_000
MAX_LABEL_LEN = 448
AUDIO_CACHE_DIR = f'{DRIVE_DIR}/audio_cache'
MANIFEST_PATH = f'{DRIVE_DIR}/manifests/train_manifest_round6b.jsonl'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints/whisper-small-round6b'
OUTPUT_DIR = f'{DRIVE_DIR}/outputs'
TEST_CACHE_DIR = f'{DRIVE_DIR}/test_parquets'
os.makedirs(TEST_CACHE_DIR, exist_ok=True)

# Balanced, memory-safe defaults for Colab Pro A100 High-RAM. Increase modestly after one successful run.
# Round 6B: lower Swahili, stronger ANV Somali/Maxatire and spontaneous-speech pressure.
SCRIPTED_TARGETS = {'swa': 2000, 'som': 4500, 'kik': 3000, 'luo': 3000, 'mas': 3500, 'kln': 3500}
UNSCRIPTED_TARGETS = {'som': 2500, 'kik': 1200, 'luo': 1200, 'mas': 2200, 'kln': 2200}
MAX_UNSCRIPTED_SHARDS = 40
ENABLE_SPEED_PERTURBATION = True
SPEED_PERTURB_LANGS = {'mas', 'kln'}
SPEED_FACTORS = [0.9, 1.1]
MAX_AUGMENTED_PER_LANG = 1200

# Run stages. Use the notebook in this order: PREPARE -> TRAIN -> INFER.
RUN_PREPARE_DATA = True
RUN_TRAINING = False
RUN_INFERENCE = False
PUSH_TO_HUB = True

MAX_STEPS = 7000
LEARNING_RATE = 6e-6
WARMUP_STEPS = 800
PER_DEVICE_TRAIN_BATCH = 16
GRADIENT_ACCUMULATION = 2
PER_DEVICE_EVAL_BATCH = 8
GEN_MAX_LENGTH = 72
GEN_BEAMS = 3

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')


In [ ]:
# CELL 3 - Normalization, audio decode, manifest helpers
ALL_PUNCT_RE = re.compile(r"[^\w\sÀ-ɏ̀-ͯḀ-ỿ'’ŋŊ]", flags=re.UNICODE)

def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFC', text).lower()
    text = ALL_PUNCT_RE.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def postprocess_submission_text(text: str) -> str:
    return normalize_text(text) or '.'

def decode_audio(audio_field):
    if isinstance(audio_field, dict) and 'array' in audio_field and audio_field['array'] is not None:
        arr = np.asarray(audio_field['array'], dtype=np.float32)
        sr = audio_field.get('sampling_rate') or 16000
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
        if sr != SAMPLE_RATE:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
        return arr.astype(np.float32, copy=False)
    raw = audio_field.get('bytes') if isinstance(audio_field, dict) else audio_field
    if isinstance(raw, bytes) and raw:
        try:
            arr, sr = sf.read(io.BytesIO(raw), dtype='float32')
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
            if sr != SAMPLE_RATE:
                arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
            return arr.astype(np.float32, copy=False)
        except Exception:
            seg = AudioSegment.from_file(io.BytesIO(raw)).set_frame_rate(SAMPLE_RATE).set_channels(1)
            return (np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0).astype(np.float32, copy=False)
    raise ValueError(f'Cannot decode audio field: {type(audio_field)}')

def audio_duration_seconds(audio_field) -> Optional[float]:
    if isinstance(audio_field, dict):
        arr = audio_field.get('array')
        if arr is not None:
            return len(arr) / float(audio_field.get('sampling_rate') or SAMPLE_RATE)
        raw = audio_field.get('bytes')
        if raw:
            try:
                info = sf.info(io.BytesIO(raw))
                return info.frames / float(info.samplerate)
            except Exception:
                return None
    return None

def speed_perturb(arr: np.ndarray, factor: float) -> np.ndarray:
    y = librosa.effects.time_stretch(arr.astype(np.float32), rate=factor)
    return y[:MAX_AUDIO_SAMPLES].astype(np.float32, copy=False)

def clip_id(prefix: str, key: str, variant: str = 'orig') -> str:
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(key))[:180]
    return f'{prefix}_{safe}_{variant}'

def save_flac_if_missing(path: str, arr: np.ndarray):
    if os.path.exists(path):
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    sf.write(path, arr[:MAX_AUDIO_SAMPLES], SAMPLE_RATE, format='FLAC', subtype='PCM_16')

def append_jsonl(path: str, rows: List[Dict[str, Any]]):
    if not rows:
        return
    with open(path, 'a', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def load_manifest(path=MANIFEST_PATH) -> List[Dict[str, Any]]:
    if not os.path.exists(path):
        return []
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def manifest_ids(path=MANIFEST_PATH):
    return {r['id'] for r in load_manifest(path)}

def ram_report(tag=''):
    vm = psutil.virtual_memory()
    print(f'{tag} RAM used={vm.used/1e9:.1f}GB available={vm.available/1e9:.1f}GB')

def collapse_ws_text(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text)).strip() or '.'

def strip_final_punct_text(text: str) -> str:
    return re.sub(r"[\s\.\,\!\?\:\;\u0964\u061f]+$", '', collapse_ws_text(text)).strip() or '.'

def strip_all_punct_text(text: str) -> str:
    return collapse_ws_text(ALL_PUNCT_RE.sub(' ', str(text)))

def lower_strip_punct_text(text: str) -> str:
    return strip_all_punct_text(str(text).lower())

def write_submission_variants(df: pd.DataFrame, primary_path: str):
    variants = {
        'raw': lambda x: collapse_ws_text(x),
        'ws': collapse_ws_text,
        'strip_final_punct': strip_final_punct_text,
        'strip_all_punct': strip_all_punct_text,
        'lower_strip_punct': lower_strip_punct_text,
    }
    written = []
    base, ext = os.path.splitext(primary_path)
    for suffix, fn in variants.items():
        out = primary_path if suffix == 'lower_strip_punct' else f'{base}_{suffix}{ext}'
        tmp = df.copy()
        tmp['transcription'] = tmp['transcription'].map(fn)
        tmp[['id', 'language', 'transcription']].to_csv(out, index=False)
        written.append(out)
    print('Submission variants written:')
    for path in written:
        print(f'  {path}')
    return written


In [ ]:
# CELL 4 - Processor, custom language tokens, label builder
processor = WhisperProcessor.from_pretrained(MODEL_ID)
feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

NEW_LANG_TOKENS = ['<|kik|>', '<|luo|>', '<|mas|>', '<|kln|>']
tokenizer.add_tokens(NEW_LANG_TOKENS, special_tokens=True)

SOT = tokenizer.convert_tokens_to_ids('<|startoftranscript|>')
TRANSCRIBE = tokenizer.convert_tokens_to_ids('<|transcribe|>')
NOTIMESTAMPS = tokenizer.convert_tokens_to_ids('<|notimestamps|>')
EOT = tokenizer.convert_tokens_to_ids('<|endoftext|>')
LANG_TOKEN = {'swa': '<|sw|>', 'som': '<|so|>', 'kik': '<|kik|>', 'luo': '<|luo|>', 'mas': '<|mas|>', 'kln': '<|kln|>'}

def build_labels(text: str, lang3: str):
    text = normalize_text(text)
    if not text:
        return None
    text_ids = tokenizer(text, add_special_tokens=False).input_ids
    labels = [SOT, tokenizer.convert_tokens_to_ids(LANG_TOKEN[lang3]), TRANSCRIBE, NOTIMESTAMPS] + text_ids + [EOT]
    if len(labels) > MAX_LABEL_LEN:
        return None
    return labels

print(f'Tokenizer size: {len(tokenizer)}')


In [ ]:
# CELL 5 - Cache harvesting functions

def add_audio_record(rows, seen, rec_id, arr, text, lang, source):
    if rec_id in seen:
        return False
    labels = build_labels(text, lang)
    if labels is None or len(arr) == 0 or len(arr) > MAX_AUDIO_SAMPLES:
        return False
    rel = f'{lang}/{rec_id}.flac'
    abs_path = os.path.join(AUDIO_CACHE_DIR, rel)
    save_flac_if_missing(abs_path, arr)
    rows.append({'id': rec_id, 'audio_path': abs_path, 'labels': labels, 'lang': lang, 'source': source, 'text': normalize_text(text)})
    seen.add(rec_id)
    return True

def count_rows(lang=None, source=None, source_prefix=None):
    rows = load_manifest()
    if lang is not None:
        rows = [r for r in rows if r['lang'] == lang]
    if source is not None:
        rows = [r for r in rows if r['source'] == source]
    if source_prefix is not None:
        rows = [r for r in rows if r['source'].startswith(source_prefix)]
    return len(rows)

def harvest_swahili(seen):
    target = SCRIPTED_TARGETS['swa']
    cached = count_rows(lang='swa')
    if cached >= target:
        print(f'Swahili already cached: {cached}/{target}')
        return
    rows = []
    made = cached
    print(f'Harvesting Swahili to {target} total records')
    for shard in range(10):
        if made >= target:
            break
        try:
            manifest_path = hf_hub_download('DigitalUmuganda/Afrivoice_Swahili', f'agriculture_swahili_train/manifest_{shard}.jsonl', repo_type='dataset', token=HF_TOKEN)
            audio_tar = hf_hub_download('DigitalUmuganda/Afrivoice_Swahili', f'agriculture_swahili_train/audio/audio_{shard}.tar.xz', repo_type='dataset', token=HF_TOKEN)
        except Exception as e:
            print(f'Swahili shard {shard} unavailable: {type(e).__name__} {str(e)[:120]}')
            break
        wanted = {}
        with open(manifest_path, encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                entry = json.loads(line)
                text = (entry.get('normalized_transcription') or entry.get('transcription') or entry.get('transcript') or entry.get('text') or '').strip()
                key = entry.get('key') or entry.get('audio_filepath') or ''
                if text and key:
                    wanted[os.path.splitext(os.path.basename(key))[0]] = text
                    wanted[str(key)] = text
        with tarfile.open(audio_tar, 'r:xz') as tar:
            for member in tar:
                if made >= target:
                    break
                ext = os.path.splitext(member.name)[1].lower()
                if ext not in ('.webm', '.wav', '.mp3', '.flac', '.ogg', '.opus'):
                    continue
                base = os.path.splitext(os.path.basename(member.name))[0]
                text = wanted.get(base) or wanted.get(member.name)
                if not text:
                    continue
                try:
                    raw = tar.extractfile(member).read()
                    arr = decode_audio({'bytes': raw})
                    if add_audio_record(rows, seen, clip_id('swa', base), arr, text, 'swa', 'scripted'):
                        made += 1
                except Exception:
                    pass
                if len(rows) >= 250:
                    append_jsonl(MANIFEST_PATH, rows); rows.clear(); ram_report('swa')
    append_jsonl(MANIFEST_PATH, rows)
    print(f'Swahili manifest rows now: {count_rows(lang="swa")}')

def download_anv_shard(shard_path: str):
    for attempt in range(3):
        try:
            return hf_hub_download('MCAA1-MSU/anv_data_ke', shard_path, repo_type='dataset', token=HF_TOKEN, local_dir=f'{LOCAL_DIR}/hf_anv')
        except Exception as e:
            print(f'Download failed {attempt+1}/3 {shard_path}: {type(e).__name__} {str(e)[:100]}')
            time.sleep(5)
    return None

def list_anv_shards():
    files = list(list_repo_files('MCAA1-MSU/anv_data_ke', repo_type='dataset', token=HF_TOKEN))
    parquets = sorted(f for f in files if f.endswith('.parquet'))
    langs = ['som', 'kik', 'luo', 'mas', 'kln']
    out = {(lang, sub): [] for lang in langs for sub in ['scripted', 'unscripted']}
    for f in parquets:
        parts = f.split('/')
        if len(parts) >= 4 and parts[0] in langs and parts[1] == 'train' and parts[2] in ('scripted', 'unscripted'):
            out[(parts[0], parts[2])].append(f)
    for k in sorted(out):
        print(f'{k[0]}/{k[1]}: {len(out[k])} shards')
    return out

def harvest_anv_bucket(lang, subtype, target, shards, seen, max_augmented=0):
    cached_primary = count_rows(lang=lang, source=subtype)
    cached_total = count_rows(lang=lang, source_prefix=subtype)
    desired_total = target + max_augmented
    if cached_primary >= target and cached_total >= desired_total:
        print(f'{lang}/{subtype} already cached: primary={cached_primary}/{target}, total={cached_total}/{desired_total}')
        return
    rows, skipped_long, skipped_bad = [], 0, 0
    primary_count = cached_primary
    total_count = cached_total
    made_aug = max(0, cached_total - cached_primary)
    per_factor_limit = max_augmented // max(1, len(SPEED_FACTORS))
    made_by_factor = {f: 0 for f in SPEED_FACTORS}
    print(f'Harvesting {lang}/{subtype}: target={target}, aug={max_augmented}')
    for shard_idx, shard_path in enumerate(shards):
        if primary_count >= target and total_count >= desired_total:
            break
        pq_path = download_anv_shard(shard_path)
        if not pq_path:
            continue
        try:
            df = pd.read_parquet(pq_path)
            tcol = next((c for c in ['transcription', 'actualSentence', 'transcript', 'text'] if c in df.columns), None)
            if tcol is None or 'audio' not in df.columns:
                continue
            for row_idx, row in df.iterrows():
                if primary_count >= target and total_count >= desired_total:
                    break
                text = (row.get(tcol) or '').strip()
                if not text:
                    continue
                dur = audio_duration_seconds(row['audio'])
                if dur is not None and dur > 30.0:
                    skipped_long += 1
                    continue
                try:
                    arr = decode_audio(row['audio'])
                except Exception:
                    skipped_bad += 1
                    continue
                key = row.get('id') or row.get('path') or row.get('audio_filepath') or f'{os.path.basename(shard_path)}_{row_idx}'
                rec_id = clip_id(f'{lang}_{subtype}', key)
                if primary_count < target and add_audio_record(rows, seen, rec_id, arr, text, lang, subtype):
                    primary_count += 1
                    total_count += 1
                if max_augmented and subtype == 'scripted' and lang in SPEED_PERTURB_LANGS and made_aug < max_augmented:
                    for factor in SPEED_FACTORS:
                        if made_by_factor[factor] >= per_factor_limit or made_aug >= max_augmented:
                            continue
                        aug_id = clip_id(f'{lang}_{subtype}', key, f'speed{factor}')
                        try:
                            aug_arr = speed_perturb(arr, factor)
                            if add_audio_record(rows, seen, aug_id, aug_arr, text, lang, f'{subtype}_speed_{factor}'):
                                made_aug += 1
                                made_by_factor[factor] += 1
                                total_count += 1
                        except Exception:
                            pass
                if len(rows) >= 250:
                    append_jsonl(MANIFEST_PATH, rows); rows.clear(); gc.collect()
        finally:
            try:
                del df
            except NameError:
                pass
            gc.collect()
        print(f'{lang}/{subtype} shard {shard_idx+1}/{len(shards)} primary={primary_count}/{target} total={total_count}/{desired_total} skipped_long={skipped_long} bad={skipped_bad} aug={made_aug}')
        ram_report(f'{lang}/{subtype}')
    append_jsonl(MANIFEST_PATH, rows)

def prepare_data_manifest(reset=False):
    if reset and os.path.exists(MANIFEST_PATH):
        os.remove(MANIFEST_PATH)
    seen = manifest_ids()
    print(f'Starting manifest rows: {len(seen)}')
    harvest_swahili(seen)
    shards = list_anv_shards()
    for lang in ['som', 'kik', 'luo', 'mas', 'kln']:
        aug = MAX_AUGMENTED_PER_LANG if ENABLE_SPEED_PERTURBATION and lang in SPEED_PERTURB_LANGS else 0
        harvest_anv_bucket(lang, 'scripted', SCRIPTED_TARGETS[lang], shards[(lang, 'scripted')], seen, max_augmented=aug)
        harvest_anv_bucket(lang, 'unscripted', UNSCRIPTED_TARGETS[lang], shards[(lang, 'unscripted')][:MAX_UNSCRIPTED_SHARDS], seen)
    rows = load_manifest()
    print(f'Total manifest rows: {len(rows)}')
    print(pd.Series([r['lang'] for r in rows]).value_counts().sort_index().to_string())
    print(pd.Series([r['source'] for r in rows]).value_counts().to_string())


In [ ]:
# CELL 6 - Run data preparation only
# First run this cell with RUN_PREPARE_DATA=True, then restart the runtime before training.
if RUN_PREPARE_DATA:
    prepare_data_manifest(reset=False)
    print('Data preparation complete. Recommended next step: Runtime -> Restart runtime, then set RUN_PREPARE_DATA=False and RUN_TRAINING=True.')


In [ ]:
# CELL 7 - Dataset, split, collator, metrics
class WhisperPathDataset(TorchDataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        return self.records[idx]

def validate_manifest_records(records):
    good, bad = [], []
    for r in records:
        if r.get('labels') and r.get('audio_path') and os.path.exists(r['audio_path']):
            good.append(r)
        else:
            bad.append(r)
    if bad:
        print(f'Dropped missing/bad rows: {len(bad)}')
    if not good:
        raise RuntimeError('No valid training rows. Run data preparation first.')
    return good

def stratified_split(records, eval_frac=0.04):
    by_bucket = {}
    for r in records:
        kind = 'unscripted' if 'unscripted' in r['source'] else 'scripted'
        by_bucket.setdefault((r['lang'], kind), []).append(r)
    rng = np.random.default_rng(SEED)
    train, eval_ = [], []
    for key, bucket in sorted(by_bucket.items()):
        bucket = list(bucket)
        rng.shuffle(bucket)
        n_eval = max(1, int(round(len(bucket) * eval_frac)))
        eval_.extend(bucket[:n_eval]); train.extend(bucket[n_eval:])
        print(f'{key}: train={len(bucket[n_eval:])} eval={n_eval}')
    rng.shuffle(train); rng.shuffle(eval_)
    return train, eval_

@dataclass
class AudioPathCollator:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Any]]):
        arrays = []
        for f in features:
            arr, sr = sf.read(f['audio_path'], dtype='float32')
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
            if sr != SAMPLE_RATE:
                arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
            arrays.append(arr[:MAX_AUDIO_SAMPLES])
        input_features = self.processor.feature_extractor(arrays, sampling_rate=SAMPLE_RATE, return_tensors='pt').input_features
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        return {'input_features': input_features, 'labels': labels}

wer_metric = None
LANG_TOKEN_ID = {tokenizer.convert_tokens_to_ids(tok): lang for lang, tok in LANG_TOKEN.items()}

def compute_metrics(pred):
    global wer_metric
    if wer_metric is None:
        wer_metric = evaluate.load('wer')
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()
    first_tok = label_ids[:, 0].copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = [postprocess_submission_text(s) for s in tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [postprocess_submission_text(s) for s in tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    metrics = {'wer': round(wer_metric.compute(predictions=pred_str, references=label_str), 4)}
    parts = []
    for tok_id, lang in LANG_TOKEN_ID.items():
        idx = [i for i, t in enumerate(first_tok) if t == tok_id]
        if idx:
            w = round(wer_metric.compute(predictions=[pred_str[i] for i in idx], references=[label_str[i] for i in idx]), 4)
            metrics[f'wer_{lang}'] = w
            parts.append(f'{lang}={w:.3f}')
    print('per-language WER: ' + ' '.join(parts), flush=True)
    return metrics


In [ ]:
# CELL 8 - Model loading and training

def load_model():
    load_from = START_FROM
    try:
        model = WhisperForConditionalGeneration.from_pretrained(load_from, token=HF_TOKEN)
        print(f'Loaded model from {load_from}')
    except Exception as e:
        print(f'Could not load {load_from}: {type(e).__name__} {str(e)[:140]}')
        model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, token=HF_TOKEN)
        print(f'Loaded base model {MODEL_ID}')
    if model.get_input_embeddings().num_embeddings != len(tokenizer):
        old_vocab = model.get_input_embeddings().num_embeddings
        model.resize_token_embeddings(len(tokenizer))
        with torch.no_grad():
            emb = model.get_input_embeddings().weight
            sw_id = tokenizer.convert_tokens_to_ids('<|sw|>')
            for tok in NEW_LANG_TOKENS:
                emb[tokenizer.convert_tokens_to_ids(tok)] = emb[sw_id].clone()
        print(f'Resized embeddings {old_vocab} -> {len(tokenizer)}')
    model.config.apply_spec_augment = True
    model.config.mask_time_prob = 0.05
    model.config.mask_time_length = 10
    model.config.mask_feature_prob = 0.03
    model.config.mask_feature_length = 10
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    model.generation_config.no_repeat_ngram_size = 3
    print(f'Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')
    return model

def train_round6():
    rows = validate_manifest_records(load_manifest())
    print(f'Total usable rows: {len(rows)}')
    print(pd.Series([r['lang'] for r in rows]).value_counts().sort_index().to_string())
    train_records, eval_records = stratified_split(rows, eval_frac=0.04)
    train_ds = WhisperPathDataset(train_records)
    eval_ds = WhisperPathDataset(eval_records)
    collator = AudioPathCollator(processor=processor, decoder_start_token_id=SOT)
    model = load_model()
    training_args = Seq2SeqTrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        lr_scheduler_type='cosine',
        label_smoothing_factor=0.05,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        fp16=True,
        optim='adamw_torch_fused' if torch.cuda.is_available() else 'adamw_torch',
        eval_strategy='steps',
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
        predict_with_generate=True,
        generation_max_length=GEN_MAX_LENGTH,
        generation_num_beams=GEN_BEAMS,
        save_steps=1000,
        eval_steps=1000,
        logging_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model='wer',
        greater_is_better=False,
        report_to=[],
        push_to_hub=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        processing_class=feature_extractor,
    )
    print('Starting training')
    trainer.train(resume_from_checkpoint=True if glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*') else None)
    trainer.save_model(CHECKPOINT_DIR)
    processor.save_pretrained(CHECKPOINT_DIR)
    print(f'Saved model to {CHECKPOINT_DIR}')
    if PUSH_TO_HUB:
        api = HfApi(token=HF_TOKEN)
        user = api.whoami()['name']
        repo_id = f'{user}/{ROUND_REPO_NAME}'
        api.create_repo(repo_id, exist_ok=True, private=True)
        trainer.model.push_to_hub(repo_id, token=HF_TOKEN)
        processor.push_to_hub(repo_id, token=HF_TOKEN)
        print(f'Pushed to https://huggingface.co/{repo_id}')
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()

if RUN_TRAINING:
    train_round6()


In [ ]:
# CELL 9 - Test cache and inference
LANG_TO_WHISPER = {'swa': 'sw', 'som': 'so', 'kik': 'kik', 'luo': 'luo', 'mas': 'mas', 'kln': 'kln'}

def cache_test_parquets():
    existing = sorted(glob.glob(os.path.join(TEST_CACHE_DIR, '**', '*.parquet'), recursive=True))
    if existing:
        print(f'Test parquets already cached: {len(existing)}')
        return existing
    test_path = kagglehub.dataset_download('digitalumuganda/anv-test-data-nt')
    downloaded = sorted(glob.glob(os.path.join(test_path, '**', '*.parquet'), recursive=True))
    print(f'Downloaded test parquets: {len(downloaded)}')
    for pf in downloaded:
        rel = os.path.relpath(pf, test_path)
        dst = os.path.join(TEST_CACHE_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(pf, dst)
    return sorted(glob.glob(os.path.join(TEST_CACHE_DIR, '**', '*.parquet'), recursive=True))

def load_inference_model():
    load_from = CHECKPOINT_DIR if os.path.exists(f'{CHECKPOINT_DIR}/config.json') else f'{HfApi(token=HF_TOKEN).whoami()["name"]}/{ROUND_REPO_NAME}'
    print(f'Loading inference model from {load_from}')
    ft_processor = WhisperProcessor.from_pretrained(load_from, token=HF_TOKEN)
    ft_model = WhisperForConditionalGeneration.from_pretrained(load_from, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32, token=HF_TOKEN).to(DEVICE)
    ft_model.eval()
    ft_model.config.forced_decoder_ids = None
    ft_model.generation_config.forced_decoder_ids = None
    ft_model.generation_config.suppress_tokens = []
    ft_model.generation_config.max_length = None
    return ft_processor, ft_model

def run_inference(beam_size=3, output_name='submission_round6b_beam3.csv'):
    parquet_files = cache_test_parquets()
    ft_processor, ft_model = load_inference_model()
    checkpoint_file = f'{OUTPUT_DIR}/{output_name.replace(".csv", "")}_checkpoint.csv'
    results = []
    done_ids = set()
    if os.path.exists(checkpoint_file):
        existing = pd.read_csv(checkpoint_file)
        results = existing.to_dict('records')
        done_ids = set(existing['id'].astype(str))
        print(f'Resuming from checkpoint rows={len(results)}')
    def transcribe_batch(arrays, language=None):
        arrays = [a[:MAX_AUDIO_SAMPLES] for a in arrays]
        inputs = ft_processor(arrays, sampling_rate=SAMPLE_RATE, return_tensors='pt').input_features.to(DEVICE)
        if DEVICE == 'cuda':
            inputs = inputs.to(torch.float16)
        gen_kw = {'max_new_tokens': 72, 'num_beams': beam_size, 'no_repeat_ngram_size': 3}
        if language:
            lid = ft_processor.tokenizer.convert_tokens_to_ids(f'<|{language}|>')
            tid = ft_processor.tokenizer.convert_tokens_to_ids('<|transcribe|>')
            nid = ft_processor.tokenizer.convert_tokens_to_ids('<|notimestamps|>')
            gen_kw['forced_decoder_ids'] = [[1, lid], [2, tid], [3, nid]]
        with torch.no_grad():
            ids = ft_model.generate(input_features=inputs, **gen_kw)
        return ft_processor.batch_decode(ids, skip_special_tokens=True)
    def safe_decode(row):
        try:
            return str(row.id), row.language, decode_audio(row.audio)
        except Exception:
            return str(row.id), row.language, None
    batch_size = 32 if DEVICE == 'cuda' else 4
    checkpoint_every_batches = 10
    batches_since_checkpoint = 0
    t0 = time.time()
    for pf_idx, pq_path in enumerate(parquet_files):
        df = pd.read_parquet(pq_path)
        df['id'] = df['id'].astype(str)
        df = df[~df['id'].isin(done_ids)]
        if df.empty:
            continue
        lang3 = df['language'].iloc[0]
        wh_lang = LANG_TO_WHISPER.get(lang3)
        rows = list(df.itertuples(index=False))
        print(f'[{pf_idx+1}/{len(parquet_files)}] {os.path.basename(pq_path)} lang={lang3} rows={len(rows)}')
        for i in range(0, len(rows), batch_size):
            chunk = rows[i:i+batch_size]
            with ThreadPoolExecutor(max_workers=4) as ex:
                decoded = list(ex.map(safe_decode, chunk))
            arrays, ids, langs = [], [], []
            for id_, lang_, arr in decoded:
                if arr is None:
                    results.append({'id': id_, 'language': lang_, 'transcription': '.'})
                    done_ids.add(id_)
                else:
                    arrays.append(arr); ids.append(id_); langs.append(lang_)
            if arrays:
                try:
                    texts = transcribe_batch(arrays, language=wh_lang)
                except Exception as e:
                    print(f'Batch error: {e}; falling back one-by-one')
                    texts = []
                    for arr in arrays:
                        try:
                            texts.append(transcribe_batch([arr], language=wh_lang)[0])
                        except Exception:
                            texts.append('.')
                for id_, lang_, text in zip(ids, langs, texts):
                    results.append({'id': id_, 'language': lang_, 'transcription': collapse_ws_text(text)})
                    done_ids.add(id_)
            batches_since_checkpoint += 1
            if batches_since_checkpoint >= checkpoint_every_batches:
                pd.DataFrame(results).to_csv(checkpoint_file, index=False)
                batches_since_checkpoint = 0
                print(f'Checkpoint saved rows={len(results)}')
        pd.DataFrame(results).to_csv(checkpoint_file, index=False)
        batches_since_checkpoint = 0
        print(f'Checkpoint saved rows={len(results)}')
    sub = pd.DataFrame(results)[['id', 'language', 'transcription']]
    out = f'{OUTPUT_DIR}/{output_name}'
    written = write_submission_variants(sub, out)
    print(f'Saved primary {out} rows={len(sub)} time={(time.time()-t0)/60:.1f} min')
    print('Primary file is lower_strip_punct because the latest Round 6 evidence favored that variant; raw/ws variants are saved beside it.')
    print(sub['language'].value_counts().to_string())
    return written

if RUN_INFERENCE:
    run_inference(beam_size=3, output_name='submission_round6b_beam3.csv')


In [ ]:
# CELL 10 - Optional beam-5 inference after beam-3 scores well
# run_inference(beam_size=5, output_name='submission_round6b_beam5.csv')
